## 1. Config and imports

In [1]:
# Install missing packages
# pip install numpy 

In [31]:
# Import and variables
from __future__ import annotations

from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = REPO_ROOT / 'prerequisites'

DATASET_END = pd.Timestamp('2026-04-17')
BACKTEST_AS_OFS = pd.to_datetime([
    '2025-09-30', '2025-10-30', '2025-11-30',
    '2025-12-15', '2026-01-15', '2026-03-15',
    '2025-12-30', '2026-01-30', '2026-02-28',
])

print('Snapshots:', list(BACKTEST_AS_OFS.strftime('%Y-%m-%d')))

Snapshots: ['2025-09-30', '2025-10-30', '2025-11-30', '2025-12-15', '2026-01-15', '2026-03-15', '2025-12-30', '2026-01-30', '2026-02-28']


In [32]:
# Datasets
raw_students = pd.read_csv(RAW_DIR / 'raw_students.csv')
raw_payments = pd.read_csv(RAW_DIR / 'raw_payments.csv')
raw_lessons = pd.read_csv(RAW_DIR / 'raw_lessons.csv')

print('students:', raw_students.shape)
print('payments:', raw_payments.shape)
print('lessons :', raw_lessons.shape)
raw_students.head(3)

students: (3000, 6)
payments: (9695, 5)
lessons : (53703, 4)


,student_id,join_ts,country_code,acquisition_channel,persona,first_subject
0,4670487,2026-02-17 09:43:33.608,ES,WoM,exam_prepper,spanish
1,4116739,2025-05-14 10:32:58.884,PL,CRM,conversationalist,english
2,4026225,2025-06-07 16:22:33.203,JP,CRM,working_professional,spanish


In [33]:
# Dataset transformations
stg_students = (
    raw_students
    .assign(
        student_id=lambda d: d['student_id'].astype('int64'),
        join_ts=lambda d: pd.to_datetime(d['join_ts']),
    )
    [['student_id', 'join_ts', 'country_code', 'acquisition_channel', 'persona', 'first_subject']]
)

stg_payments = (
    raw_payments
    .assign(
        payment_id=lambda d: d['payment_id'].astype('int64'),
        student_id=lambda d: d['student_id'].astype('int64'),
        payment_ts=lambda d: pd.to_datetime(d['payment_ts']),
        hours=lambda d: d['hours'].astype('int64'),
        price_per_hour_usd=lambda d: d['price_per_hour_usd'].astype('float64'),
    )
    .assign(payment_date=lambda d: d['payment_ts'].dt.date)
    [['payment_id', 'student_id', 'payment_date', 'payment_ts', 'hours', 'price_per_hour_usd']]
)

stg_lessons = (
    raw_lessons
    .assign(
        lesson_id=lambda d: d['lesson_id'].astype('int64'),
        student_id=lambda d: d['student_id'].astype('int64'),
        booking_ts=lambda d: pd.to_datetime(d['booking_ts']),
        hours_booked=lambda d: d['hours_booked'].astype('float64'),
    )
    [['lesson_id', 'student_id', 'booking_ts', 'hours_booked']]
)

stg_payments.head(3)

,payment_id,student_id,payment_date,payment_ts,hours,price_per_hour_usd
0,150000000,4670487,2026-02-18,2026-02-18 02:35:11.741,6,17.50
1,150000001,4670487,2026-03-18,2026-03-18 02:35:11.596,6,17.50
2,150000002,4670487,2026-04-15,2026-04-15 02:35:11.447,6,17.50


In [34]:
# Mirror payments with cycles macros
payments_with_cycle = (
    stg_payments
    .sort_values(['student_id', 'payment_ts', 'payment_id'])
    .assign(cycle_start_ts=lambda d: d['payment_ts'])
)
next_ts = payments_with_cycle.groupby('student_id')['payment_ts'].shift(-1)
payments_with_cycle['cycle_end_ts'] = next_ts.fillna(
    payments_with_cycle['payment_ts'] + pd.Timedelta(days=28)
)
payments_with_cycle = payments_with_cycle.rename(columns={'hours': 'hours_purchased'})

payments_with_cycle[['payment_id', 'student_id', 'cycle_start_ts', 'cycle_end_ts', 'hours_purchased']].head(5)

,payment_id,student_id,cycle_start_ts,cycle_end_ts,hours_purchased
2337,150002337,4000425,2026-01-24 21:27:00.656,2026-02-21 21:27:00.489,6
2338,150002338,4000425,2026-02-21 21:27:00.489,2026-03-21 21:27:00.390,6
2339,150002339,4000425,2026-03-21 21:27:00.390,2026-04-18 21:27:00.390,8
792,150000792,4000599,2025-05-29 16:34:27.009,2025-06-26 16:34:27.064,12
793,150000793,4000599,2025-06-26 16:34:27.064,2025-07-24 16:34:27.742,12


In [35]:
def hours_booked_per_cycle(
    payments_cycles: pd.DataFrame,
    lessons: pd.DataFrame,
    as_of_ts: pd.Timestamp | None = None,
) -> pd.DataFrame:
    pc = payments_cycles[['payment_id', 'student_id', 'cycle_start_ts', 'cycle_end_ts']]
    merged = lessons.merge(pc, on='student_id')
    in_cycle = (merged['booking_ts'] >= merged['cycle_start_ts']) & (merged['booking_ts'] < merged['cycle_end_ts'])
    if as_of_ts is not None:
        in_cycle &= merged['booking_ts'] <= as_of_ts
    grouped = (
        merged.loc[in_cycle].groupby('payment_id', as_index=False)['hours_booked'].sum()
        .rename(columns={'hours_booked': 'hours_booked'})
    )
    return grouped

hours_final = hours_booked_per_cycle(payments_with_cycle, stg_lessons).rename(
    columns={'hours_booked': 'hours_booked_in_cycle'}
)
print('cycles with bookings:', len(hours_final), '/', len(payments_with_cycle))

cycles with bookings: 9622 / 9695


In [36]:
actuals = (
    payments_with_cycle
    .merge(hours_final, on='payment_id', how='left')
    .assign(hours_booked_in_cycle=lambda d: d['hours_booked_in_cycle'].fillna(0))
    .assign(
        hours_unbooked=lambda d: np.clip(d['hours_purchased'] - d['hours_booked_in_cycle'], 0, None),
    )
    .assign(actual_breakage_usd=lambda d: d['hours_unbooked'] * d['price_per_hour_usd'])
    [['payment_id', 'cycle_start_ts', 'cycle_end_ts', 'hours_purchased', 'price_per_hour_usd',
      'hours_booked_in_cycle', 'hours_unbooked', 'actual_breakage_usd']]
)
actuals.head(3)

,payment_id,cycle_start_ts,cycle_end_ts,hours_purchased,price_per_hour_usd,hours_booked_in_cycle,hours_unbooked,actual_breakage_usd
0,150002337,2026-01-24 21:27:00.656,2026-02-21 21:27:00.489,6,11.00,4.50,1.50,16.50
1,150002338,2026-02-21 21:27:00.489,2026-03-21 21:27:00.390,6,11.00,4.00,2.00,22.00
2,150002339,2026-03-21 21:27:00.390,2026-04-18 21:27:00.390,8,11.00,5.00,3.00,33.00


In [37]:
# Define cohort types
def closed_cycle_pct_unbooked(
    payments_cycles: pd.DataFrame,
    lessons: pd.DataFrame,
    students: pd.DataFrame,
    as_of_ts: pd.Timestamp,
) -> pd.DataFrame:
    final = hours_booked_per_cycle(payments_cycles, lessons).rename(
        columns={'hours_booked': 'hours_booked_in_cycle'}
    )
    df = (
        payments_cycles
        .merge(students[['student_id', 'country_code', 'persona', 'acquisition_channel']], on='student_id')
        .merge(final, on='payment_id', how='left')
        .assign(hours_booked_in_cycle=lambda d: d['hours_booked_in_cycle'].fillna(0))
        .query('cycle_end_ts <= @as_of_ts and hours_purchased > 0')
        .assign(
            pct_unbooked=lambda d: np.clip(d['hours_purchased'] - d['hours_booked_in_cycle'], 0, None) / d['hours_purchased']
        )
    )
    return df


def build_cohort_rates(closed: pd.DataFrame) -> dict:
    specific = (
        closed.groupby(['country_code', 'persona', 'acquisition_channel'], as_index=False)['pct_unbooked']
        .mean().rename(columns={'pct_unbooked': 'specific_rate'})
    )
    generic = (
        closed.groupby(['country_code'], as_index=False)['pct_unbooked']
        .mean().rename(columns={'pct_unbooked': 'generic_rate'})
    )
    global_rate = closed['pct_unbooked'].mean()
    return {'specific': specific, 'generic': generic, 'global': global_rate}


sample_closed = closed_cycle_pct_unbooked(payments_with_cycle, stg_lessons, stg_students, BACKTEST_AS_OFS[-1])
print(f'closed cycles at {BACKTEST_AS_OFS[-1].date()}: {len(sample_closed):,}  |  global pct_unbooked = {sample_closed["pct_unbooked"].mean():.4f}')

closed cycles at 2026-02-28: 7,023  |  global pct_unbooked = 0.2753


In [38]:
# Define comparable cycles
def open_at_as_of(
    payments_cycles: pd.DataFrame,
    lessons: pd.DataFrame,
    students: pd.DataFrame,
    as_of_ts: pd.Timestamp,
    dataset_end_ts: pd.Timestamp,
) -> pd.DataFrame:
    booked_to_asof = hours_booked_per_cycle(payments_cycles, lessons, as_of_ts=as_of_ts).rename(
        columns={'hours_booked': 'hours_booked_to_as_of_date'}
    )
    open_df = (
        payments_cycles
        .merge(students[['student_id', 'country_code', 'persona', 'acquisition_channel']], on='student_id')
        .merge(booked_to_asof, on='payment_id', how='left')
        .assign(hours_booked_to_as_of_date=lambda d: d['hours_booked_to_as_of_date'].fillna(0))
        .query('cycle_end_ts > @as_of_ts and cycle_end_ts <= @dataset_end_ts and hours_purchased > 0')
        .assign(
            hours_remaining=lambda d: np.clip(d['hours_purchased'] - d['hours_booked_to_as_of_date'], 0, None),
            utilization_pct=lambda d: d['hours_booked_to_as_of_date'] / d['hours_purchased'],
        )
    )
    return open_df

sample_open = open_at_as_of(payments_with_cycle, stg_lessons, stg_students, BACKTEST_AS_OFS[-1], DATASET_END)
print(f'eval-eligible open cycles at {BACKTEST_AS_OFS[-1].date()}: {len(sample_open):,}')

eval-eligible open cycles at 2026-02-28: 1,797


```
if utilization_pct > 1 - pct_unbooked:
    breakage = hours_remaining * price
else:
    breakage = hours_purchased * pct_unbooked * price
```

In [39]:
# Define different strategies to get unbooked pct: comparing against global rate or more specific cohorts
STRATEGIES = ('hierarchy', 'generic_only', 'global_only')


def attach_rates(open_df: pd.DataFrame, rates: dict) -> pd.DataFrame:
    df = open_df.merge(rates['specific'], on=['country_code', 'persona', 'acquisition_channel'], how='left')
    df = df.merge(rates['generic'], on=['country_code'], how='left')
    df['global_rate'] = rates['global']
    return df


def predict(df: pd.DataFrame, strategy: str) -> pd.Series:
    if strategy == 'hierarchy':
        rate = df['specific_rate'].fillna(df['generic_rate']).fillna(df['global_rate'])
    elif strategy == 'generic_only':
        rate = df['generic_rate'].fillna(df['global_rate'])
    elif strategy == 'global_only':
        rate = df['global_rate']
    else:
        raise ValueError(strategy)
    high_pace = df['utilization_pct'] > (1 - rate)
    est_usd = np.where(
        high_pace,
        df['hours_remaining'] * df['price_per_hour_usd'],
        np.clip(df['hours_purchased'], 0, None) * rate * df['price_per_hour_usd'],
    )
    return pd.Series(est_usd, index=df.index, name=f'{strategy}_estimated_usd')

In [40]:
def evaluate_snapshot(as_of_ts: pd.Timestamp) -> pd.DataFrame:
    closed = closed_cycle_pct_unbooked(payments_with_cycle, stg_lessons, stg_students, as_of_ts)
    if len(closed) == 0:
        return pd.DataFrame()
    rates = build_cohort_rates(closed)
    open_df = open_at_as_of(payments_with_cycle, stg_lessons, stg_students, as_of_ts, DATASET_END)
    if open_df.empty:
        return pd.DataFrame()
    open_df = attach_rates(open_df, rates)
    open_df = open_df.merge(
        actuals[['payment_id', 'actual_breakage_usd']], on='payment_id', how='left'
    )

    rows = []
    for strat in STRATEGIES:
        est = predict(open_df, strat)
        err = est - open_df['actual_breakage_usd']
        rows.append({
            'as_of_date': as_of_ts.date(),
            'cohort_strategy': strat,
            'payment_count': len(open_df),
            'total_estimated_usd': est.sum(),
            'total_actual_usd': open_df['actual_breakage_usd'].sum(),
            'mae_usd': err.abs().mean(),
            'wape': err.abs().sum() / max(open_df['actual_breakage_usd'].sum(), 1e-9),
            'mean_error_usd': err.mean(),
        })
    return pd.DataFrame(rows)

per_snapshot = pd.concat([evaluate_snapshot(t) for t in BACKTEST_AS_OFS], ignore_index=True)
per_snapshot

,as_of_date,cohort_strategy,payment_count,total_estimated_usd,total_actual_usd,mae_usd,wape,mean_error_usd
0,2025-09-30,hierarchy,7196,"282,270.24","261,425.50",22.64,0.62,2.90
1,2025-09-30,generic_only,7196,"279,624.99","261,425.50",20.96,0.58,2.53
2,2025-09-30,global_only,7196,"281,567.57","261,425.50",20.94,0.58,2.80
3,2025-10-30,hierarchy,6413,"249,116.42","232,623.00",22.58,0.62,2.57
4,2025-10-30,generic_only,6413,"247,624.12","232,623.00",20.89,0.58,2.34
5,2025-10-30,global_only,6413,"249,070.81","232,623.00",20.92,0.58,2.56
6,2025-11-30,hierarchy,5446,"209,287.54","195,708.75",22.01,0.61,2.49
7,2025-11-30,generic_only,5446,"209,602.88","195,708.75",20.73,0.58,2.55
8,2025-11-30,global_only,5446,"210,583.49","195,708.75",20.76,0.58,2.73
9,2025-12-15,hierarchy,4914,"186,856.21","175,983.50",21.75,0.61,2.21


In [41]:
def pooled_predictions() -> pd.DataFrame:
    frames = []
    for t in BACKTEST_AS_OFS:
        closed = closed_cycle_pct_unbooked(payments_with_cycle, stg_lessons, stg_students, t)
        if closed.empty:
            continue
        rates = build_cohort_rates(closed)
        open_df = open_at_as_of(payments_with_cycle, stg_lessons, stg_students, t, DATASET_END)
        if open_df.empty:
            continue
        open_df = attach_rates(open_df, rates).merge(
            actuals[['payment_id', 'actual_breakage_usd']], on='payment_id', how='left'
        )
        for strat in STRATEGIES:
            est = predict(open_df, strat)
            frames.append(pd.DataFrame({
                'as_of_date': t.date(),
                'payment_id': open_df['payment_id'].values,
                'cohort_strategy': strat,
                'estimated_usd': est.values,
                'actual_usd': open_df['actual_breakage_usd'].values,
            }))
    return pd.concat(frames, ignore_index=True)

pred_long = pooled_predictions().assign(
    err=lambda d: d['estimated_usd'] - d['actual_usd'],
    abs_err=lambda d: (d['estimated_usd'] - d['actual_usd']).abs(),
)

pooled = (
    pred_long
    .groupby('cohort_strategy', as_index=False)
    .agg(
        payment_count=('payment_id', 'count'),
        total_estimated_usd=('estimated_usd', 'sum'),
        total_actual_usd=('actual_usd', 'sum'),
        mae_usd=('abs_err', 'mean'),
        mean_error_usd=('err', 'mean'),
    )
)
wape_per_strat = pred_long.groupby('cohort_strategy').apply(
    lambda g: g['abs_err'].sum() / max(g['actual_usd'].sum(), 1e-9)
)
pooled['wape'] = pooled['cohort_strategy'].map(wape_per_strat)
pooled = pooled.sort_values('wape').reset_index(drop=True)
pooled

C:\Users\HP\AppData\Local\Temp\ipykernel_16352\2074655127.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  wape_per_strat = pred_long.groupby('cohort_strategy').apply(


,cohort_strategy,payment_count,total_estimated_usd,total_actual_usd,mae_usd,mean_error_usd,wape
0,generic_only,38041,"1,452,670.06","1,355,096.75",20.51,2.56,0.58
1,global_only,38041,"1,458,964.22","1,355,096.75",20.54,2.73,0.58
2,hierarchy,38041,"1,450,426.94","1,355,096.75",21.55,2.51,0.60
